In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/merged_less_obs.h5ad')

In [ ]:
adata

In [ ]:
genebody = sc.read_h5ad(f'{projdir}/h5ad/genebody/merged_norm.h5ad')

In [ ]:
genebody

In [ ]:
genebody.obs.index = [x.split('_indexed')[0] for x in genebody.obs.index]

In [ ]:
g_mask = genebody.obs.index.isin(adata.obs.index)
a_mask = adata.obs.index.isin(genebody.obs.index)

In [ ]:
final_genebody = genebody[g_mask]
final_adata = adata[a_mask]

In [ ]:
final_adata.obs.index == final_genebody.obs.index

### wait I can just take the umap lol

In [ ]:
umap = pd.DataFrame(index=final_adata.obs.index, data=final_adata.obsm['X_umap'])

In [ ]:
final_genebody.obsm['X_umap'] = umap.reindex(final_genebody.obs.index).values

In [ ]:
final_genebody.obs = final_adata.obs.reindex(final_genebody.obs.index)

In [ ]:
sc.pl.umap(final_genebody, color=['time', 'line'])

In [ ]:
final_genebody

In [ ]:
map_path = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations/ensembl_to_symbol.tsv'
ensg_to_symbol = pd.read_csv(map_path, sep='\t', header=0, index_col=0)

In [ ]:
gene_dict = dict(zip(ensg_to_symbol['gene'], ensg_to_symbol['symbol']))

In [ ]:
final_genebody.var['symbol'] = [gene_dict[x] for x in final_genebody.var.index]

In [ ]:
final_genebody.var['gene'] = final_genebody.var.index

In [ ]:
final_genebody.var.index = final_genebody.var['symbol']

In [ ]:
final_genebody.var.index

In [ ]:
sc.pl.umap(final_genebody, color=['FAM138A'], vmin=0.5, vmax=1)

In [ ]:
final_genebody.var.index.name = 'final_gene'

In [ ]:
final_genebody.write_h5ad(f'{projdir}/h5ad/genebody/chrom100k_umap.h5ad')

### load in justin clusters

In [ ]:
indir = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3_Testrun3/YS3.v4_Metadata/'
degs = pd.read_csv(f'{indir}/YS3.v4.NameCluster.DiffExp.top50.markergenes.df.txt', sep='\t', header=0, index_col=0)

In [ ]:
degs['coarse'] = [x.split('_')[0] for x in degs['cluster']]
degs['coarse'].replace({'early1' : 'early', 'early2' : 'early', 'early3' : 'early',
                        'early4' : 'early', 'early5' : 'early',
                        'ips1' : 'ips', 'ips2' : 'ips',
                       'ips3' : 'ips', 'ips4' : 'ips', 'ips5' : 'ips', }, inplace=True)
degs['coarse'].unique()

In [ ]:
tmp_degs = degs[degs['coarse'] == 'ips']
sub_clusts = sorted(tmp_degs['cluster'].unique())
top_n = 20
for clust in sub_clusts:
    print(clust)
    final_degs = tmp_degs[tmp_degs['cluster'] == clust]
    sc.pl.umap(final_genebody, color=final_degs['gene'][:top_n], vmin=0.5, vmax=1, ncols=5)

In [ ]:
tmp_degs = degs[degs['coarse'] == 'fail1']
sub_clusts = sorted(tmp_degs['cluster'].unique())
top_n = 20
for clust in sub_clusts:
    print(clust)
    final_degs = tmp_degs[tmp_degs['cluster'] == clust]
    sc.pl.umap(final_genebody, color=final_degs['gene'][:top_n], vmin=0.5, vmax=1, ncols=5)

In [ ]:
tmp_degs = degs[degs['coarse'] == 'early']
sub_clusts = sorted(tmp_degs['cluster'].unique())
top_n = 20
for clust in sub_clusts:
    print(clust)
    final_degs = tmp_degs[tmp_degs['cluster'] == clust]
    sc.pl.umap(final_genebody, color=final_degs['gene'][:top_n], vmin=0.5, vmax=1, ncols=5)